<a href="https://colab.research.google.com/github/coksvictoria/ACVAE/blob/main/synthesis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ucimlrepo

In [ ]:
from ucimlrepo import fetch_ucirepo

import pandas as pd
import numpy as np
from math import log2

In [ ]:
# List of selected health dataset IDs
datasets = {
    "adult":2,
    "pima_diabetes": 17,
    "heart_disease": 45,
    "breast_cancer": 17,
    "parkinsons": 176,
}

loaded_datasets = {}

for name, dataset_id in datasets.items():
    print(f"Loading {name}...")
    data = fetch_ucirepo(id=dataset_id)

    X = data.data.features
    y = data.data.targets

    # Combine for easier analysis
    df = pd.concat([X, y], axis=1)
    loaded_datasets[name] = df
    print(f"✅ Loaded {name} with shape {df.shape}")

print("\nAll datasets loaded successfully!")

Loading adult...
✅ Loaded adult with shape (48842, 15)
Loading pima_diabetes...
✅ Loaded pima_diabetes with shape (569, 31)
Loading heart_disease...
✅ Loaded heart_disease with shape (303, 14)
Loading breast_cancer...
✅ Loaded breast_cancer with shape (569, 31)
Loading parkinsons...
✅ Loaded parkinsons with shape (748, 5)

All datasets loaded successfully!


In [ ]:
from itertools import combinations

def is_deterministic(df, cols_input, col_target):
    """
    Check if col_target is 100% deterministically defined by cols_input.
    cols_input can be one or more columns (list of strings).
    """
    temp = df[cols_input + [col_target]].dropna()
    seen = {}
    for row in temp.itertuples(index=False, name=None):
        input_key = row[:-1]  # tuple of all input columns
        target_value = row[-1]
        if input_key in seen:
            if seen[input_key] != target_value:
                return False
        else:
            seen[input_key] = target_value
    return True

def find_deterministic_combinations(df):
    """
    Detect 1→1 deterministic relationships between columns in a dataframe.
    Avoids reverse duplicates.

    Returns:
        List of tuples: ((input_cols), target_col)
    """
    deterministic_pairs = []
    seen_sets = set()  # To avoid reverse duplicates

    cols = df.columns

    for i in range(len(cols)):
        for j in range(len(cols)):
            if i == j:
                continue

            inp = cols[i]
            target = cols[j]

            # Check 1→1 determinism
            if is_deterministic(df, [inp], target):
                # Create a frozenset of the two columns to avoid reverse duplicates
                col_set = frozenset([inp, target])
                if col_set not in seen_sets:
                    deterministic_pairs.append(((inp,), target))
                    seen_sets.add(col_set)

    return deterministic_pairs

In [ ]:
# Apply deterministic detection to all loaded datasets
all_deterministic_results = {}

for name, df in loaded_datasets.items():
    print(f"\nAnalyzing deterministic relationships in {name} (shape: {df.shape})...")

    # Run deterministic detection (you can adjust max_inputs)
    deterministic_relations = find_deterministic_combinations(df)

    all_deterministic_results[name] = deterministic_relations

    if deterministic_relations:
        print(f"✅ Found {len(deterministic_relations)} deterministic relationships:")
        for inp, target in deterministic_relations:
            print(f"   {inp} -> {target}")
    else:
        print("No deterministic relationships found.")

print("\nAll datasets analyzed successfully!")


Analyzing deterministic relationships in adult (shape: (48842, 15))...
✅ Found 1 deterministic relationships:
   ('education',) -> education-num

Analyzing deterministic relationships in pima_diabetes (shape: (569, 31))...
No deterministic relationships found.

Analyzing deterministic relationships in heart_disease (shape: (303, 14))...
No deterministic relationships found.

Analyzing deterministic relationships in breast_cancer (shape: (569, 31))...
No deterministic relationships found.

Analyzing deterministic relationships in parkinsons (shape: (748, 5))...
✅ Found 1 deterministic relationships:
   ('Frequency',) -> Monetary

All datasets analyzed successfully!


In [ ]:
# Make a copy to avoid modifying the original loaded_datasets
pruned_datasets = {}

for name, df in loaded_datasets.items():
    # Get deterministic targets for this dataset
    deterministic_pairs = all_deterministic_results.get(name, [])
    targets_to_drop = [target for inp, target in deterministic_pairs]

    # Drop target columns if they exist in the dataframe
    pruned_df = df.drop(columns=[c for c in targets_to_drop if c in df.columns])
    pruned_datasets[name] = pruned_df

    print(f"{name}: dropped {len(targets_to_drop)} deterministic column(s) -> shape now {pruned_df.shape}")

adult: dropped 1 deterministic column(s) -> shape now (48842, 14)
pima_diabetes: dropped 0 deterministic column(s) -> shape now (569, 31)
heart_disease: dropped 0 deterministic column(s) -> shape now (303, 14)
breast_cancer: dropped 0 deterministic column(s) -> shape now (569, 31)
parkinsons: dropped 1 deterministic column(s) -> shape now (748, 4)


In [ ]:
restored_datasets = {}

for name, df_pruned in pruned_datasets.items():
    restored_df = df_pruned.copy()
    for inp_cols, target_col in all_deterministic_results.get(name, []):
        mapping_df = loaded_datasets[name].set_index(list(inp_cols))
        mapping = mapping_df[target_col].to_dict()

        if len(inp_cols) == 1:
            restored_df[target_col] = restored_df[inp_cols[0]].map(mapping)
        else:
            restored_df[target_col] = restored_df[list(inp_cols)].apply(lambda row: mapping[tuple(row)], axis=1)

    restored_datasets[name] = restored_df

,age,workclass,fnlwgt,education,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income,education-num
0,39,State-gov,77516,Bachelors,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K,13
1,50,Self-emp-not-inc,83311,Bachelors,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K,13
2,38,Private,215646,HS-grad,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K,9
3,53,Private,234721,11th,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K,7
4,28,Private,338409,Bachelors,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K,13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48837,39,Private,215419,Bachelors,Divorced,Prof-specialty,Not-in-family,White,Female,0,0,36,United-States,<=50K.,13
48838,64,NaN,321403,HS-grad,Widowed,NaN,Other-relative,Black,Male,0,0,40,United-States,<=50K.,9
48839,38,Private,374983,Bachelors,Married-civ-spouse,Prof-specialty,Husband,White,Male,0,0,50,United-States,<=50K.,13
48840,44,Private,83891,Bachelors,Divorced,Adm-clerical,Own-child,Asian-Pac-Islander,Male,5455,0,40,United-States,<=50K.,13
